In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
os.getenv("OPENAI_API_KEY") is not None

True

# Test de la construction du profil de l'étudiant en amont du RAG

In [5]:
from openai import OpenAI
client = OpenAI()

system = (
        "Tu es un assistant qui analyse des messages de lycéens pour "
        "remplir un profil structuré StudentProfile. "
        "Tu ne fais PAS de psychologie, tu ne déduis PAS l'origine, "
        "le genre ou la personnalité. "
        "Tu dois renvoyer UNIQUEMENT un objet JSON avec les clés : "
        "type_formation, is_apprentissage, max_frais_scolarite, commune, "
        "domains_interet, matieres_aimees, matieres_evitees, "
        "preference_rythme, preference_travail, niveau_idee_orientation. "
        "Les champs peuvent être null ou des listes vides si l'information "
        "n'est pas présente dans le message."
    )

msg = "J'aime les maths, l'informatique et je préfère travailler en équipe. Je ne sais pas si je veux de l'apprentissage."

response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": system},
        {"role": "user", "content": msg},
    ],
    max_output_tokens=150,
)

raw = response.output[0].content[0].text
print(raw)

```json
{
  "type_formation": null,
  "is_apprentissage": null,
  "max_frais_scolarite": null,
  "commune": null,
  "domains_interet": [],
  "matieres_aimees": ["maths", "informatique"],
  "matieres_evitees": [],
  "preference_rythme": null,
  "preference_travail": ["équipe"],
  "niveau_idee_orientation": null
}
```


In [3]:
from src.backend.profile_from_text import infer_profile_from_text

msg = "J'aime les maths, l'informatique et je préfère travailler en équipe. Je ne sais pas si je veux de l'apprentissage."
profile = infer_profile_from_text(msg)
profile

{'type_formation': None,
 'is_apprentissage': None,
 'max_frais_scolarite': None,
 'commune': None,
 'domains_interet': [],
 'matieres_aimees': ['maths', 'informatique'],
 'matieres_evitees': [],
 'preference_rythme': None,
 'preference_travail': ['équipe'],
 'niveau_idee_orientation': None}

# Test de la fonction `student_orientation`

In [3]:
from src.backend.chat_pipeline import student_orientation

msg = (
    "J'aime les maths et l'informatique, je fais du tennis en club "
    "et j'aime travailler en équipe. Je ne sais pas si je veux de l'apprentissage."
)

reponse = student_orientation(msg)
print(reponse)

/Users/maelioviau-vonfelt/Desktop/Masters/Epitech Data/Cours/M2/Mémoires/Consulting - StudAI/etudIA/env_etudia/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8903.11it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tu aimes les maths et l'informatique, et tu préfères travailler en équipe, ce qui t'ouvre plusieurs pistes de formation. 

Une possibilité est de te tourner vers les formations des écoles d'ingénieurs, qui valorisent souvent le travail en projet et l'esprit d'équipe. Cependant, cette option a des frais de scolarité plus élevés. 

Si tu cherches une formation qui intègre l'apprentissage, il y a des licences en santé à Vannes ou Nice, qui respectent ton budget et pourraient combiner théorie et pratique. 

Cela te permettrait de découvrir différentes facettes des domaines qui t'intéressent. N'hésite pas à explorer ces options pour voir celle qui te correspond le mieux !


## Amélioration du prompt dans `call_llm_advisor`

In [4]:
msg = (
    "J'aime les maths et l'informatique, je fais du tennis en club "
    "et j'aime travailler en équipe. Je ne sais pas si je veux de l'apprentissage."
)

reponse = student_orientation(msg)
print(reponse)

Étant donné que vous aimez les maths et l'informatique, plusieurs formations peuvent vous correspondre. Vous pourriez envisager des études en santé ou une licence, qui respectent votre budget tout en proposant éventuellement de l'apprentissage. Par exemple, il y a des formations à Vannes et Nice qui pourraient répondre à vos attentes. 

Si vous êtes ouvert à d'autres types de formation, les écoles d'ingénieurs à Courbevoie, bien que plus coûteuses, pourraient également vous intéresser, surtout si vous aimez le travail en équipe. Prenez le temps de réfléchir à ce qui vous attire le plus. Ces suggestions sont des pistes à explorer, pas des décisions définitives.


Résultat: Plus court le temps de traitement de 41,3s à 6,7s